In [ ]:
import os
import pandas as pd
import numpy as np

hitph_data_dir = "../data"
sorted(os.listdir(hitph_data_dir))

---

level 4 first

In [ ]:
df_level4 = pd.read_csv(os.path.join(
    hitph_data_dir,
    "Hi-TpH-level-IV.csv"
))
print(df_level4.shape)

df_level4['STAPLER'] = "yes"
df_level4.loc[:25457, 'STAPLER'] = 'no'
df_level4.head(3)

In [ ]:
df_level4 = df_level4.dropna(subset=['antigen.epitope', 'beta.cdr3', "alpha.cdr3", "hla.short.seq",
                               "alpha.vseq.reconstructed",  "beta.vseq.reconstructed"])
df_level4.rename(columns={"antigen.epitope":"pep", "hla.short.seq":"hla", 
                       "beta.cdr3":"beta_cdr3", "alpha.cdr3":"alpha_cdr3",
                       "beta.vseq.reconstructed":"beta",
                       "alpha.vseq.reconstructed":"alpha",}, inplace=True)

df_level4['id'] = df_level4['pep'] + '/' + df_level4['beta_cdr3']
df_level4 = df_level4.drop_duplicates(ignore_index=True)

print(df_level4.shape)

In [ ]:
# length restriction
df_level4 = df_level4[df_level4['pep'].str.len() <= 10]
df_level4 = df_level4[df_level4['pep'].str.len() >= 8]
print(df_level4.shape)

df_level4 = df_level4[df_level4['alpha_cdr3'].str.len() <= 19]
df_level4 = df_level4[df_level4['alpha_cdr3'].str.len() >= 9]
print(df_level4.shape)

df_level4 = df_level4[df_level4['beta_cdr3'].str.len() <= 19]
df_level4 = df_level4[df_level4['beta_cdr3'].str.len() >= 9]
print(df_level4.shape)

df_level4 = df_level4[df_level4['alpha'].str.len() <= 121]
df_level4 = df_level4[df_level4['alpha'].str.len() >= 105]
print(df_level4.shape)

df_level4 = df_level4[df_level4['beta'].str.len() <= 121]
df_level4 = df_level4[df_level4['beta'].str.len() >= 109]
print(df_level4.shape)

In [ ]:
df_level4["ab"] = df_level4['alpha'] + '/' + df_level4['beta']
df_level4["ab_cdr3"] = df_level4['alpha_cdr3'] + '/' + df_level4['beta_cdr3']

df_level4 = df_level4.drop_duplicates(subset=['pep', 'hla', 'ab'], ignore_index=True)
print(len(df_level4))
df_level4 = df_level4.drop_duplicates(subset=['pep', 'hla', 'alpha', 'beta'], ignore_index=True)
print(len(df_level4))

In [ ]:
pep_num_df = df_level4['pep'].value_counts().to_frame()
print("Number of peptides:", len(pep_num_df), ", Number of samples:", pep_num_df['count'].sum())

# exclude STAPLER samples, which are not in Level I-III
df_level4_ex = df_level4[df_level4['STAPLER']=='no']
pep_num_df = df_level4_ex['pep'].value_counts().to_frame()
print("Number of peptides:", len(pep_num_df), ", Number of samples:", pep_num_df['count'].sum())

# long tail peptides
max_num = 50
pep_num_df = pep_num_df[pep_num_df['count']<=max_num]
pep_num_df = pep_num_df.sample(frac=1, random_state=42)     # shuffle
print("Number of peptides:", len(pep_num_df), ", Number of samples:", pep_num_df['count'].sum())

# randomly select 100 long tail peptides for unseen set
num_selected_pep = 100
selected_pep_num_df = pep_num_df.iloc[:num_selected_pep]
print("Number of peptides:", len(selected_pep_num_df), ", Number of samples:", selected_pep_num_df['count'].sum())

In [ ]:
dataset_save_dir = "./level4"
os.makedirs(dataset_save_dir, exist_ok=True)

# Save unique TCRs to candidates_pool
tcr_list = df_level4['ab'].unique().tolist()
print(len(tcr_list))
np.save(os.path.join(dataset_save_dir, "tcr2candidates_pools.npy"), tcr_list)

tcr_list = df_level4['ab_cdr3'].unique().tolist()
print(len(tcr_list))
np.save(os.path.join(dataset_save_dir, "tcr2candidates_pools_ab_cdr3.npy"), tcr_list)

ab_cdr3_vjgene = df_level4['alpha.v']+'+'+df_level4['alpha.j']+'+'+df_level4['alpha_cdr3']+\
                '&'+df_level4['beta.v']+'+'+df_level4['beta.j']+'+'+df_level4['beta_cdr3']
tcr_list = ab_cdr3_vjgene.unique().tolist()
print(len(tcr_list))
np.save(os.path.join(dataset_save_dir, "tcr2candidates_pools_ab_cdr3_vjgene.npy"), tcr_list)

# tcr_list = df_level4['beta'].unique().tolist()
# print(len(tcr_list))
# np.save(os.path.join(dataset_save_dir, "tcr2candidates_pools_beta.npy"), tcr_list)

# tcr_list = df_level4['beta_cdr3'].unique().tolist()
# print(len(tcr_list))
# np.save(os.path.join(dataset_save_dir, "tcr2candidates_pools_beta_cdr3.npy"), tcr_list)

In [ ]:
# add label
df_level4['label'] = 1

# save selected long tail unseen peps to 'Unseen set'
selected_pep = selected_pep_num_df.index.to_list()
external_df = df_level4[df_level4['pep'].isin(selected_pep)].reset_index(drop=True)
external_id = external_df['id'].to_list()
external_df = external_df[['pep', 'hla', 'ab', 'hla.allele', 
                           'ab_cdr3', 'beta', 'beta_cdr3',
                           'alpha.v', 'alpha.j', 'beta.v', 'beta.j', 
                           'label']]

# split the remaining data into train/valid/test set in the ratio of 8:1:1
df_level4_left = df_level4[~df_level4['pep'].isin(selected_pep)].reset_index(drop=True)
df_level4_left = df_level4_left[['pep', 'hla', 'ab', 'hla.allele', 
                                 'ab_cdr3', 'beta', 'beta_cdr3',
                                 'alpha.v', 'alpha.j', 'beta.v', 'beta.j', 
                                 'label']]

df_shuffled = df_level4_left.sample(frac=1, random_state=42).reset_index(drop=True)

total_samples = len(df_shuffled)
train_samples = int(0.8 * total_samples)
val_samples = int(0.1 * total_samples)
test_samples = total_samples - train_samples - val_samples

train_df = df_shuffled.iloc[:train_samples]
val_df = df_shuffled.iloc[train_samples:train_samples + val_samples]
test_df = df_shuffled.iloc[train_samples + val_samples:]

train_df.to_csv(os.path.join(dataset_save_dir, "train_data_fold0.csv"), index=False)
val_df.to_csv(os.path.join(dataset_save_dir, "valid_data_fold0.csv"), index=False)
test_df.to_csv(os.path.join(dataset_save_dir, "test_data_fold0.csv"), index=False)
external_df.to_csv(os.path.join(dataset_save_dir, "unseen_data.csv"), index=False)
print(len(train_df), len(val_df), len(test_df), len(external_df), external_df['pep'].nunique())

---

level 1

In [ ]:
df_level1 = pd.read_csv(os.path.join(
    hitph_data_dir,
    "Hi-TpH-level-I.csv"
))
print(df_level1.shape)
df_level1.head()

In [ ]:
df_level1 = df_level1.dropna(subset=['antigen.epitope', 'beta.cdr3'])
df_level1.rename(columns={"antigen.epitope":"pep", "beta.cdr3":"beta"}, inplace=True)

df_level1['id'] = df_level1['pep'] + '/' + df_level1['beta']
df_level1 = df_level1.drop_duplicates(ignore_index=True)
print(df_level1.shape)

In [ ]:
# length restriction
df_level1 = df_level1[df_level1['pep'].str.len() <= 15]
df_level1 = df_level1[df_level1['pep'].str.len() >= 8]
print(len(df_level1))

df_level1 = df_level1[df_level1['beta'].str.len() <= 19]
df_level1 = df_level1[df_level1['beta'].str.len() >= 9]
print(len(df_level1))

In [ ]:
dataset_save_dir = "./level1"
os.makedirs(dataset_save_dir, exist_ok=True)

# Save unique TCRs to candidates_pool
tcr_list = df_level1['beta'].unique().tolist()
print(len(tcr_list))
np.save(os.path.join(dataset_save_dir, "tcr2candidates_pools.npy"), tcr_list)

In [ ]:
# add label
df_level1['label'] = 1

# save selected long tail unseen peps(in level4) to 'Unseen set'
external_df = df_level1[df_level1['pep'].isin(selected_pep)]
external_df = external_df[['pep', 'beta', 'label']]

# split the remaining data into train/valid/test set in the ratio of 8:1:1
df_level1_left = df_level1[~df_level1['pep'].isin(selected_pep)].reset_index(drop=True)
df_level1_left = df_level1_left[['pep', 'beta', 'label']]
print(len(df_level1_left))

df_shuffled = df_level1_left.sample(frac=1, random_state=42).reset_index(drop=True)

total_samples = len(df_shuffled)
train_samples = int(0.8 * total_samples)
val_samples = int(0.1 * total_samples)
test_samples = total_samples - train_samples - val_samples

train_df = df_shuffled.iloc[:train_samples]
val_df = df_shuffled.iloc[train_samples:train_samples + val_samples]
test_df = df_shuffled.iloc[train_samples + val_samples:]

train_df.to_csv(os.path.join(dataset_save_dir, "train_data_fold0.csv"), index=False)
val_df.to_csv(os.path.join(dataset_save_dir, "valid_data_fold0.csv"), index=False)
test_df.to_csv(os.path.join(dataset_save_dir, "test_data_fold0.csv"), index=False)
external_df.to_csv(os.path.join(dataset_save_dir, "unseen_data.csv"), index=False)
print(len(train_df), len(val_df), len(test_df), len(external_df), external_df['pep'].nunique())

---

level2

In [ ]:
df_level2 = pd.read_csv(os.path.join(
    hitph_data_dir,
    "Hi-TpH-level-II.csv"
))
print(df_level2.shape)
df_level2.head(3)

In [ ]:
df_level2 = df_level2.dropna(subset=['antigen.epitope', 'beta.cdr3',"hla.short.seq"])
df_level2.rename(columns={"antigen.epitope":"pep", "hla.short.seq":"hla", "beta.cdr3":"beta"}, inplace=True)

df_level2['id'] = df_level2['pep'] + '/' + df_level2['beta']
df_level2 = df_level2.drop_duplicates(ignore_index=True)
print(df_level2.shape)

In [ ]:
# length restriction
df_level2 = df_level2[df_level2['beta'].str.len() <= 19]
df_level2 = df_level2[df_level2['beta'].str.len() >= 9]
print(df_level2.shape)

df_level2 = df_level2[df_level2['pep'].str.len() <= 10]
df_level2 = df_level2[df_level2['pep'].str.len() >= 8]
print(df_level2.shape)

In [ ]:
dataset_save_dir = "./level2"
os.makedirs(dataset_save_dir, exist_ok=True)

# Save unique TCRs to candidates_pool
tcr_list = df_level2['beta'].unique().tolist()
print(len(tcr_list))
np.save(os.path.join(dataset_save_dir, "tcr2candidates_pools.npy"), tcr_list)

In [ ]:
# add label
df_level2['label'] = 1

# save selected long tail unseen peps(in level4) to 'Unseen set'
external_df = df_level2[df_level2['pep'].isin(selected_pep)]
external_df = external_df[['pep', 'beta', 'hla', 'hla.allele', 'label']]

# split the remaining data into train/valid/test set in the ratio of 8:1:1
df_level2_left = df_level2[~df_level2['pep'].isin(selected_pep)].reset_index(drop=True)
df_level2_left = df_level2_left[['pep', 'beta', 'hla', 'hla.allele', 'label']]
print(len(df_level2_left))

df_shuffled = df_level2_left.sample(frac=1, random_state=42).reset_index(drop=True)

total_samples = len(df_shuffled)
train_samples = int(0.8 * total_samples)
val_samples = int(0.1 * total_samples)
test_samples = total_samples - train_samples - val_samples

train_df = df_shuffled.iloc[:train_samples]
val_df = df_shuffled.iloc[train_samples:train_samples + val_samples]
test_df = df_shuffled.iloc[train_samples + val_samples:]

train_df.to_csv(os.path.join(dataset_save_dir, "train_data_fold0.csv"), index=False)
val_df.to_csv(os.path.join(dataset_save_dir, "valid_data_fold0.csv"), index=False)
test_df.to_csv(os.path.join(dataset_save_dir, "test_data_fold0.csv"), index=False)
external_df.to_csv(os.path.join(dataset_save_dir, "unseen_data.csv"), index=False)
print(len(train_df), len(val_df), len(test_df), len(external_df), external_df['pep'].nunique())

---

level3

In [ ]:
df_level3 = pd.read_csv(os.path.join(
    hitph_data_dir,
    "Hi-TpH-level-III.csv"
))
print(df_level3.shape)
df_level3.head(3)

In [ ]:
df_level3 = df_level3.dropna(subset=['antigen.epitope', 'beta.cdr3', "alpha.cdr3", "hla.short.seq"])
df_level3.rename(columns={"antigen.epitope":"pep", "hla.short.seq":"hla", 
                       "beta.cdr3":"beta", "alpha.cdr3":"alpha"}, inplace=True)

df_level3['id'] = df_level3['pep'] + '/' + df_level3['beta']
df_level3 = df_level3.drop_duplicates(ignore_index=True)
print(df_level3.shape)

In [ ]:
# length restriction
df_level3 = df_level3[df_level3['pep'].str.len() <= 10]
df_level3 = df_level3[df_level3['pep'].str.len() >= 8]
print(df_level3.shape)

df_level3 = df_level3[df_level3['alpha'].str.len() <= 19]
df_level3 = df_level3[df_level3['alpha'].str.len() >= 9]
print(df_level3.shape)

df_level3 = df_level3[df_level3['beta'].str.len() <= 19]
df_level3 = df_level3[df_level3['beta'].str.len() >= 9]
print(df_level3.shape)

In [ ]:
df_level3["ab"] = df_level3['alpha'] + '/' + df_level3['beta']

df_level3 = df_level3.drop_duplicates(subset=['pep', 'hla', 'ab'], ignore_index=True)
print(len(df_level3))
df_level3 = df_level3.drop_duplicates(subset=['pep', 'hla', 'alpha', 'beta'], ignore_index=True)
print(len(df_level3))

In [ ]:
dataset_save_dir = "./level3"
os.makedirs(dataset_save_dir, exist_ok=True)

# Save unique TCRs to candidates_pool
tcr_list = df_level3['ab'].unique().tolist()
print(len(tcr_list))
np.save(os.path.join(dataset_save_dir, "tcr2candidates_pools.npy"), tcr_list)

# tcr_list = df_level3['beta'].unique().tolist()
# print(len(tcr_list))
# np.save(os.path.join(dataset_save_dir, "tcr2candidates_pools_beta.npy"), tcr_list)

In [ ]:
# add label
df_level3['label'] = 1

# save selected long tail unseen peps(in level4) to 'Unseen set'
external_df = df_level3[df_level3['pep'].isin(selected_pep)]
external_df = external_df[['pep', 'hla', 'ab', 'hla.allele', 'beta', 'label']]

# split the remaining data into train/valid/test set in the ratio of 8:1:1
df_level3_left = df_level3[~df_level3['pep'].isin(selected_pep)].reset_index(drop=True)
df_level3_left = df_level3_left[['pep', 'hla', 'ab', 'hla.allele', 'beta', 'label']]
print(len(df_level3_left))

df_shuffled = df_level3_left.sample(frac=1, random_state=42).reset_index(drop=True)

total_samples = len(df_shuffled)
train_samples = int(0.8 * total_samples)
val_samples = int(0.1 * total_samples)
test_samples = total_samples - train_samples - val_samples

train_df = df_shuffled.iloc[:train_samples]
val_df = df_shuffled.iloc[train_samples:train_samples + val_samples]
test_df = df_shuffled.iloc[train_samples + val_samples:]

train_df.to_csv(os.path.join(dataset_save_dir, "train_data_fold0.csv"), index=False)
val_df.to_csv(os.path.join(dataset_save_dir, "valid_data_fold0.csv"), index=False)
test_df.to_csv(os.path.join(dataset_save_dir, "test_data_fold0.csv"), index=False)
external_df.to_csv(os.path.join(dataset_save_dir, "unseen_data.csv"), index=False)
print(len(train_df), len(val_df), len(test_df), len(external_df), external_df['pep'].nunique())